<a href="https://colab.research.google.com/github/giripriyansenthilkumar/Innomatics_Task/blob/main/Innomatics_task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers accelerate datasets tokenizers
!pip install transformers==4.37.2
!pip install accelerate==0.26.1
!pip install datasets==2.16.1
!pip install seqeval
!pip uninstall -y peft

Found existing installation: transformers 4.37.2
Uninstalling transformers-4.37.2:
  Successfully uninstalled transformers-4.37.2
Found existing installation: accelerate 0.26.1
Uninstalling accelerate-0.26.1:
  Successfully uninstalled accelerate-0.26.1
Found existing installation: tokenizers 0.15.2
Uninstalling tokenizers-0.15.2:
  Successfully uninstalled tokenizers-0.15.2
  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.37.2-py3-none-any.whl (8.4 MB)
Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
peft 0.18.1 requires accelerate>=0.21.0, which is 

In [2]:
import transformers, datasets, accelerate

print(transformers.__version__)
print(datasets.__version__)
print(accelerate.__version__)


4.37.2
2.16.1
0.26.1


In [3]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)
from seqeval.metrics import f1_score


In [4]:
dataset = load_dataset("conll2003")
print(dataset)


/usr/local/lib/python3.12/dist-packages/datasets/load.py:1429: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [5]:
label_list = dataset["train"].features["pos_tags"].feature.names
print(label_list)


['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        padding="max_length",   # ✅ FIX
        max_length=128,         # ✅ FIX
        is_split_into_words=True
    )

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(example["pos_tags"][word_idx])
        else:
            label_ids.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs


In [15]:
tokenized_datasets = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs"
)

In [11]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]

    true_predictions = [
        [label_list[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return {
        "f1": f1_score(true_labels, true_predictions)
    }

In [16]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.756600,0.258089,0.910584
2,0.205200,0.229813,0.917258
3,0.165500,0.223066,0.919956


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

TrainOutput(global_step=2634, training_loss=0.30355453201560423, metrics={'train_runtime': 572.7311, 'train_samples_per_second': 73.548, 'train_steps_per_second': 4.599, 'total_flos': 1376994620968704.0, 'train_loss': 0.30355453201560423, 'epoch': 3.0})

In [19]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

{'eval_loss': 0.2230658382177353, 'eval_f1': 0.9199558075442835, 'eval_runtime': 14.0669, 'eval_samples_per_second': 231.039, 'eval_steps_per_second': 14.502, 'epoch': 3.0}


In [21]:
import torch

def predict(sentence):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    # ✅ FIX: move inputs to same device as model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs).logits
    predictions = outputs.argmax(dim=2)

    predicted_labels = [label_list[p.item()] for p in predictions[0]]

    print("\nPrediction:\n")
    for token, label in zip(tokens, predicted_labels[1:len(tokens)+1]):
        print(f"{token} -> {label}")


In [23]:
string=input("Enter a sentence: ")
predict(string)

Enter a sentence: John works at Google in California

Prediction:

John -> NNP
works -> VBZ
at -> IN
Google -> NNP
in -> IN
California -> NNP


In [26]:
chunk_label_list = dataset["train"].features["chunk_tags"].feature.names
print(chunk_label_list)


['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [27]:
def tokenize_and_align_chunk_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(example["chunk_tags"][word_idx])  # ✅ chunk labels
        else:
            label_ids.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs


In [28]:
chunk_tokenized = dataset.map(tokenize_and_align_chunk_labels)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [29]:
chunk_model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(chunk_label_list),
    id2label={i: l for i, l in enumerate(chunk_label_list)},
    label2id={l: i for i, l in enumerate(chunk_label_list)}
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
chunk_trainer = Trainer(
    model=chunk_model,
    args=training_args,
    train_dataset=chunk_tokenized["train"],
    eval_dataset=chunk_tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [31]:
chunk_trainer.train()

Epoch,Training Loss,Validation Loss
1,0.469100,0.202375
2,0.166500,0.181796
3,0.132100,0.174396


Checkpoint destination directory ./results/checkpoint-878 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results/checkpoint-1756 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results/checkpoint-2634 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=2634, training_loss=0.21829277435395422, metrics={'train_runtime': 575.3327, 'train_samples_per_second': 73.215, 'train_steps_per_second': 4.578, 'total_flos': 1376397560805120.0, 'train_loss': 0.21829277435395422, 'epoch': 3.0})

In [32]:
chunk_results = chunk_trainer.evaluate()
print(chunk_results)

{'eval_loss': 0.1743955761194229, 'eval_runtime': 14.0809, 'eval_samples_per_second': 230.809, 'eval_steps_per_second': 14.488, 'epoch': 3.0}


In [34]:
import torch

def predict_both(sentence):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True,
        padding=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # POS
    model.to(device)
    pos_inputs = {k: v.to(device) for k, v in inputs.items()}
    pos_outputs = model(**pos_inputs).logits
    pos_preds = pos_outputs.argmax(dim=2)

    # Chunk
    chunk_model.to(device)
    chunk_inputs = {k: v.to(device) for k, v in inputs.items()}
    chunk_outputs = chunk_model(**chunk_inputs).logits
    chunk_preds = chunk_outputs.argmax(dim=2)

    print("\nFinal Prediction:\n")
    for i, token in enumerate(tokens):
        pos = label_list[pos_preds[0][i+1].item()]
        chunk = chunk_label_list[chunk_preds[0][i+1].item()]
        print(f"{token:12} POS: {pos:6} | Chunk: {chunk}")

In [35]:
predict_both("John works at Google in California")



Final Prediction:

John         POS: NNP    | Chunk: B-NP
works        POS: VBZ    | Chunk: B-VP
at           POS: IN     | Chunk: B-PP
Google       POS: NNP    | Chunk: B-NP
in           POS: IN     | Chunk: B-PP
California   POS: NNP    | Chunk: B-NP
